# fastcausal Quickstart

This notebook demonstrates the basic capabilities of **fastcausal** for causal discovery analysis.

**Five lines to your first causal graph:**

```python
from fastcausal import FastCausal
fc = FastCausal()
df = fc.load_sample("boston")          # EMA dataset: alcohol, sleep, mood
df_prep = fc.standardize(fc.add_lag_columns(df))
results, graph = fc.run_search(df_prep, algorithm="gfci")
fc.show_graph(graph)
```

## Setup

In [ ]:
from fastcausal import FastCausal

fc = FastCausal()

## 1. Load data

The bundled `"boston"` dataset is an ecological momentary assessment (EMA) dataset with daily measurements of alcohol use, sleep, and mood variables.

In [ ]:
df = fc.load_sample("boston")
print(f"Shape: {df.shape}")
df.head()

## 2. Prepare data

For time-series data, we add **lagged columns** (yesterday's values) and **standardize** all variables.

In [ ]:
# Add lagged columns (suffix '_lag')
df_lag = fc.add_lag_columns(df)

# Standardize to zero mean, unit variance
df_prep = fc.standardize(df_lag)

print(f"Columns: {df_prep.columns.tolist()}")
df_prep.head()

## 3. Create temporal knowledge

Since lag variables represent the *past*, they can only be **causes**, never effects, of current-day variables.

In [ ]:
knowledge = fc.create_lag_knowledge(df.columns)
knowledge

## 4. Run causal discovery

Run GFCI (Greedy Fast Causal Inference) — a score-based algorithm that handles latent confounders.

In [ ]:
results, graph = fc.run_search(
    df_prep,
    algorithm="gfci",
    alpha=0.05,
    penalty_discount=1.0,
    knowledge=knowledge,
)

print(f"Edges found: {results['num_edges']}")
print("\n".join(results["edges"]))

## 5. Visualize the graph

In [ ]:
fc.show_graph(graph)

In [ ]:
# Directed edges only
fc.show_graph(graph, directed_only=True)

## 6. SEM results

When `run_sem=True` (the default), a structural equation model is fit and edge weights are available.

In [ ]:
if results.get("sem_results"):
    print(results["sem_results"]["estimates"])

## Next steps

- **`ema_analysis.ipynb`** — full EMA workflow: node styling, multi-graph comparison, stability search
- **`batch_project.ipynb`** — config-driven batch analysis with `fastcausal run --config`
- **CLI**: `fastcausal analyze data.csv --algorithm gfci`